Read the silver layer

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f

spark= SparkSession.builder.getOrCreate()

silver_path = "/Volumes/workspace/default/airlines/silver/flights/"
silver_df= spark.read.format("delta").load(silver_path)

print(f" Silver rows loaded : {silver_df.count():,}")

 Silver rows loaded : 538,062


In [0]:
#Gold Table 1: Route KPIs
route_kpis = (silver_df.groupBy("Origin", "Dest", "Route", "Month").agg(
    f.count("*").alias("TotalFlights"),
    f.sum("Cancelled").alias("TotalCancelledFlights"),
    f.avg("ArrDelay").alias("AvgArrDelay"),
    f.avg("DepDelay").alias("AvgDepDelay"),
    f.avg("Distance").alias("AvgDistance"),
    f.sum(f.when(f.col("ArrDelay")<= 15,1)
          .otherwise(0)).alias("OnTimeFlights")
    )
    .withColumn("OTP_Pct",
                f.round(f.col("OnTimeFlights")/f.col("TotalFlights")*100,1))
    .withColumn("CancelRate_PCT", f.round(f.col("TotalCancelledFlights")/f.col("TotalFlights")*100,1))
    )

print(" Route KPIs ready!")
print(f" Route KPIs rows : {route_kpis.count():,}")

display(route_kpis.orderBy(f.col("TotalFlights").desc()).limit(10))

 Route KPIs ready!
 Route KPIs rows : 5,578


Origin,Dest,Route,Month,TotalFlights,TotalCancelledFlights,AvgArrDelay,AvgDepDelay,AvgDistance,OnTimeFlights,OTP_Pct,CancelRate_PCT
OGG,HNL,OGG-HNL,1,1039,21,4.615014436958614,2.762271414821944,100.0,862,83.0,2.0
HNL,OGG,HNL-OGG,1,1039,18,5.817131857555341,5.719923002887391,100.0,873,84.0,1.7
LAX,LAS,LAX-LAS,1,964,13,12.943983402489627,16.347510373443985,236.0,725,75.2,1.3
LAS,LAX,LAS-LAX,1,958,15,15.551148225469728,16.178496868475992,236.0,677,70.7,1.6
BOS,DCA,BOS-DCA,1,918,13,2.372549019607843,5.332244008714597,399.0,754,82.1,1.4
DCA,BOS,DCA-BOS,1,916,17,3.611353711790393,8.52292576419214,399.0,755,82.4,1.9
LAX,SFO,LAX-SFO,1,899,17,13.774193548387096,12.519466073414906,337.0,621,69.1,1.9
SFO,LAX,SFO-LAX,1,884,19,6.916289592760181,12.588235294117647,337.0,671,75.9,2.1
LGA,ORD,LGA-ORD,1,879,21,-1.9772468714448237,13.193401592718999,733.0,713,81.1,2.4
ORD,LGA,ORD-LGA,1,877,21,13.714937286202964,14.18472063854048,733.0,596,68.0,2.4


In [0]:
#Filter out the routes with less than 10 flights (statistically unreliable)

print(f"*** Routes before filter: {route_kpis.count():,}")
route_kpis =route_kpis.filter(f.col("TotalFlights")>=10)

#Overwrite Gold route_kpis
route_kpis.write.format("delta").mode("overwrite").save("/Volumes/workspace/default/airlines/gold/route_kpis/")

print(" route_kpis updated with minimum flight filter")
print(f"*** Routes after filter: {route_kpis.count():,}")

display(route_kpis)


*** Routes before filter: 5,578
 route_kpis updated with minimum flight filter
*** Routes after filter: 4,766


Origin,Dest,Route,Month,TotalFlights,TotalCancelledFlights,AvgArrDelay,AvgDepDelay,AvgDistance,OnTimeFlights,OTP_Pct,CancelRate_PCT
BWI,JFK,BWI-JFK,1,79,1,1.2278481012658229,6.113924050632911,184.0,69,87.3,1.3
GRR,DTW,GRR-DTW,1,123,2,-0.4715447154471545,2.707317073170732,120.0,110,89.4,1.6
ATL,XNA,ATL-XNA,1,113,1,9.353982300884956,10.63716814159292,589.0,98,86.7,0.9
JFK,MCI,JFK-MCI,1,62,0,4.983870967741935,17.838709677419356,1113.0,46,74.2,0.0
MQT,DTW,MQT-DTW,1,31,1,5.064516129032258,4.709677419354839,349.0,25,80.6,3.2
GSP,DTW,GSP-DTW,1,20,0,-2.05,-1.1,508.0,19,95.0,0.0
ATL,CSG,ATL-CSG,1,81,4,10.049382716049383,15.567901234567902,83.0,66,81.5,4.9
DSM,MSP,DSM-MSP,1,87,3,0.12643678160919541,6.264367816091954,232.0,77,88.5,3.4
JFK,CLE,JFK-CLE,1,133,1,11.12781954887218,15.887218045112782,425.0,104,78.2,0.8
MSP,MEM,MSP-MEM,1,78,2,5.166666666666667,6.730769230769231,700.0,65,83.3,2.6


In [0]:
#Display the column OPT_pct
display(route_kpis.select("OTP_Pct"))
#Gold Table 2: Carrier KPIs


OTP_Pct
87.3
89.4
86.7
74.2
80.6
95.0
81.5
88.5
78.2
83.3


In [0]:
#Carrier Scorecard
#Total flights, Average arrival delays, On time flights,Cancelled flights, Total weather delay, Total carrier delay, Total NAS delay, OTP%

carrier_kpis = (silver_df.groupby("Reporting_Airline").agg(f.count("*").alias("TotalFlights"), f.avg("ArrDelay").alias("AvgArrDelay"), f.sum(f.when(f.col("ArrDelay")<=15,1).otherwise(0)).alias("Total_On_Time_Flights"), f.sum("Cancelled").alias("Cancelled_flights"), f.sum("WeatherDelay").alias("Total_Weather_Delay"), f.sum("CarrierDelay").alias("Total_Carrier_Delay"), f.sum("NASDelay").alias("Total_NAS_Delay"),f.round(f.col("Total_On_Time_Flights")/f.col("TotalFlights")*100,1).alias("OTP_pct")))
display(carrier_kpis.orderBy(f.col("TotalFlights").desc()).limit(10))

Reporting_Airline,TotalFlights,AvgArrDelay,Total_On_Time_Flights,Cancelled_flights,Total_Weather_Delay,Total_Carrier_Delay,Total_NAS_Delay,OTP_pct
WN,112426,4.7358973902833865,91399,3234,22582.0,378124.0,205560.0,81.3
DL,75016,4.694425189292951,60448,585,55620.0,419243.0,196699.0,80.6
AA,74773,7.6494322817059635,58849,1411,43841.0,307640.0,289136.0,78.7
UA,56596,7.354689377341155,44058,413,23761.0,256366.0,221111.0,77.8
OO,50224,8.536177922905384,39518,1668,146197.0,381523.0,41807.0,78.7
YX,24474,-1.6699763013810573,20662,386,12236.0,45812.0,102984.0,84.4
B6,23220,9.229888027562446,17330,194,2976.0,169821.0,95934.0,74.6
NK,21858,12.866822216122243,15837,507,8518.0,128862.0,164620.0,72.5
AS,19799,3.1330875296732157,15857,280,7574.0,58276.0,64573.0,80.1
MQ,18834,6.404959116491452,14738,482,22668.0,43630.0,79569.0,78.3


In [0]:
# Day of Week Heatmap
day_of_week_heatmap = (silver_df.groupBy("DayName", "DayOfWeek", "Origin").agg(f.avg("ArrDelay").alias("AvgArrDelay"), f.count("*").alias("Total_Flights")))
display(day_of_week_heatmap.orderBy(f.col("DayOfWeek").desc()).limit(10))
    

DayName,DayOfWeek,Origin,AvgArrDelay,Total_Flights
Saturday,7,BTV,-4.3283582089552235,67
Saturday,7,KOA,0.21693121693121692,189
Saturday,7,HPN,-11.341463414634147,123
Saturday,7,LIT,0.967032967032967,91
Saturday,7,CVG,-9.136807817589576,307
Saturday,7,TVC,-2.04,25
Saturday,7,MOT,1.4,20
Saturday,7,GFK,-4.4375,16
Saturday,7,SFO,7.271369294605809,1205
Saturday,7,BUR,-0.35909090909090907,220


In [0]:
#Save all 3 gold tables
#gold_path = "/Volumes/workspace/default/airlines/gold/flights/"

#Table1: route_kpis
route_path = "/Volumes/workspace/default/airlines/gold/route_kpis"
route_kpis.write.format("delta").mode("overwrite").save(route_path)
print("-route_kpis saved")

spark.sql(f"""
          CREATE TABLE IF NOT EXISTS route_kpis
          USING DELTA
          AS select* from DELTA.`/Volumes/workspace/default/airlines/gold/route_kpis`
          """)


#Table2: carrier_kpis
carrier_path ="/Volumes/workspace/default/airlines/gold/carrier_kpis"
carrier_kpis.write.format("delta").mode("overwrite").save(carrier_path)
print("- carrier_kpis saved!")

spark.sql(f"""
          CREATE TABLE IF NOT EXISTS carrier_kpis
          USING DELTA
          AS select* from DELTA.`/Volumes/workspace/default/airlines/gold/carrier_kpis`
          """)

#Table3: day_of_week_heatmap
day_of_week_path = "/Volumes/workspace/default/airlines/gold/day_of_week_heatmap"
day_of_week_heatmap.write.format("delta").mode("overwrite").save(day_of_week_path)
print("Day of week heatmap data saved!")

spark.sql(f"""
          Create TABLE if not exists day_of_week_heatmap
          Using delta
          as select* from delta.`/Volumes/workspace/default/airlines/gold/day_of_week_heatmap`
          """)

          
print("gold layer saved!")
print("gold LAYER COMPLETE! ")

-route_kpis saved
- carrier_kpis saved!
Day of week heatmap data saved!
gold layer saved!
gold LAYER COMPLETE! 


In [0]:
#Check for negative OPT values
route_kpis_df = spark.read.format("delta").load("/Volumes/workspace/default/airlines/gold/route_kpis")
print("Routes with negative OTP%: ")
display(route_kpis_df.filter(route_kpis_df.OTP_Pct<0)
        .select("Route", "TotalFlights", "OnTimeFlights", "OTP_Pct")
        .orderBy("OTP_Pct")
        .limit(10))

Routes with negative OTP%: 


Route,TotalFlights,OnTimeFlights,OTP_Pct
